## Imports

In [ ]:
import os
import re
import glob
import shutil
import numpy as np
from IPython.utils import io

## Excluir Pastas

In [ ]:
base_dir = '..'
folders_to_delete = ['datasets', 'inputs', 'models', 'outputs', 'results']

In [ ]:
for folder in folders_to_delete:
    folder_path = os.path.join(base_dir, folder)
    if os.path.exists(folder_path):
        shutil.rmtree(folder_path)
        print(f"Pasta '{folder}' excluída")
    else:
        print(f"Pasta '{folder}' não encontrada")

## Execução

In [ ]:
iterations = 10 # Número de execuções

In [ ]:
if not os.getcwd().endswith('src'):
    os.chdir('src')

In [ ]:
current_dir = os.getcwd()

for iteration in range(iterations):
    with io.capture_output() as captured:
        %run 1.dataset.ipynb
        %run 2.train.ipynb
        %run 3.inputs.ipynb
        %run 4.inference.ipynb
        %run 5.fuzzy.ipynb
        %run 6.results.ipynb
    
    print(f"Pipeline completo - Iteração {iteration+1}/{iterations} concluída")

os.chdir(current_dir)
print(f"Total: {iterations} pipelines completos")

# Métricas

In [ ]:
result_folders = glob.glob('../results/run_*')

In [ ]:
accuracies = []
precisions = []
recalls = []
f1_scores = []

In [ ]:
for folder in result_folders[:iterations]:
    metrics_file = os.path.join(folder, 'metrics.txt')
    
    if os.path.exists(metrics_file):
        with open(metrics_file, 'r') as f:
            content = f.read()
            
            acc_match = re.search(r'Accuracy:\s+(\d+\.\d+)%', content)
            prec_match = re.search(r'Precision:\s+(\d+\.\d+)%', content)
            rec_match = re.search(r'Recall:\s+(\d+\.\d+)%', content)
            f1_match = re.search(r'F1:\s+(\d+\.\d+)%', content)
            
            if acc_match:
                accuracies.append(float(acc_match.group(1)))
            if prec_match:
                precisions.append(float(prec_match.group(1)))
            if rec_match:
                recalls.append(float(rec_match.group(1)))
            if f1_match:
                f1_scores.append(float(f1_match.group(1)))

In [ ]:
with open('../results/metrics.txt', 'w') as f:
    f.write("Metrics\n")
    f.write("-" * 25 + "\n")
    f.write(f"Accuracy:   {np.mean(accuracies):.2f}% (±{np.std(accuracies):.2f})\n")
    f.write(f"Precision:  {np.mean(precisions):.2f}% (±{np.std(precisions):.2f})\n")
    f.write(f"Recall:     {np.mean(recalls):.2f}% (±{np.std(recalls):.2f})\n")
    f.write(f"F1:         {np.mean(f1_scores):.2f}% (±{np.std(f1_scores):.2f})\n")

    print("\Métricas salvas em: ../results/metrics.txt")

In [ ]:
print("Metrics")
print("-" * 25)
print(f"Accuracy:        {np.mean(accuracies):.2f}% (±{np.std(accuracies):.2f})")
print(f"Precision:       {np.mean(precisions):.2f}% (±{np.std(precisions):.2f})")
print(f"Recall:          {np.mean(recalls):.2f}% (±{np.std(recalls):.2f})")
print(f"F1:              {np.mean(f1_scores):.2f}% (±{np.std(f1_scores):.2f})")